# ECLAT bằng Julia

**Mục tiêu.** Từ vài giao dịch, tự tính một nhánh ECLAT rồi kiểm nghiệm bằng hai backend của nhóm. Sau đó đọc số liệu hiệu năng của bốn bộ dữ liệu FIM.

Một **giao dịch** là một tập item, có TID riêng kể cả khi trùng nội dung với dòng khác. Một **itemset** $X$ có support tuyệt đối $\sigma(X)=|\{t:X\subseteq t\}|$. Support tương đối là $\sigma(X)/n$ với $n$ giao dịch. Minsup $\theta$ cho ngưỡng nguyên $\lceil n\theta\rceil$ trên dữ liệu không rỗng. ECLAT xuất mọi tập đủ ngưỡng và support. Luật kết hợp $A\Rightarrow B$ cần bước sinh luật và lọc confidence sau khi khai thác tập phổ biến.

| Thành phần | Dùng để làm gì |
|---|---|
| Julia Base, `BitVector`, thư viện chuẩn `Random`, `SHA` | ECLAT do nhóm cài đặt, tập con seed cố định và kiểm hash |
| Python chuẩn, Git, curl | Tổ chức chạy, tải và chuẩn hóa input FIMI, đọc CSV |
| Matplotlib và IPython | Vẽ lại từ CSV đã đo và hiển thị hình, bảng trong notebook |
| Java 21+ và SPMF 2.64b | Oracle độc lập trên cùng input và ngưỡng đếm |

Nhóm dựa trên biểu diễn dọc và lớp tương đương tiền tố của Zaki. Code dựng cặp đầu bằng giao singleton và duyệt mỗi lớp con ngay theo DFS. [Bài báo gốc](https://doi.org/10.1109/69.846291) · [Mã Julia](../src/algorithm/)

## Môi trường và thư mục kết quả

Dùng Colab CPU hoặc Jupyter từ clone đầy đủ. Chọn **Run all** từ đầu. Ô khởi tạo in commit, phiên bản môi trường và tạo thư mục kết quả riêng. Ở Colab nó clone public `main`. Trên máy cá nhân cần Julia ≥1.9, Python ≥3.11, Java ≥21 và curl, theo [README](../README.md).

In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path(tempfile.mkdtemp(prefix="eclat-", dir="/content"))
    URL = "https://github.com/phuctran33/-DM-Eclat.git"
    REF = "main"
    subprocess.run(["git", "clone", "--single-branch", "--branch", REF, URL, str(ROOT)], check=True)
else:
    ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                 if (path / "src/EclatProject.jl").is_file()), None)
    if ROOT is None:
        raise RuntimeError("Open the notebook from the project checkout.")

sys.path.insert(0, str(ROOT / "notebooks"))
from eclat_setup import csv_rows, run, run_julia, setup, show_table

OUTPUT, JULIA, JAVA = setup(ROOT, IN_COLAB, "demo")

## 1. Tự tính trên dữ liệu toy

Dữ liệu ngang giữ mỗi hàng là một giao dịch. Biểu diễn dọc giữ $T(i)$, tập TID của từng item. Khi ghép hai tập trong cùng lớp tiền tố, $T(X\cup Y)=T(X)\cap T(Y)$, nên một phép giao cho support của tập mới. Với `basic.txt`, 40% của năm dòng là hai TID. Xem TID 1 và TID 5, hai giỏ cùng nội dung được tính riêng.

In [ ]:
from collections import Counter, defaultdict
from fractions import Fraction
from math import ceil

toy = ROOT / "data/toy/basic.txt"
transactions = [list(map(int, row.split())) for row in toy.read_text(encoding="utf-8").splitlines()]
vertical = defaultdict(set)
for tid, items in enumerate(transactions, 1):
    for item in items:
        vertical[item].add(tid)

threshold = ceil(Fraction("0.4") * len(transactions))
show_table(["TID", "Item IDs"], enumerate(transactions, 1))
show_table(["Item", "TID list", "Support", "Giữ"],
           [(item, sorted(tids), len(tids), "có" if len(tids) >= threshold else "không")
            for item, tids in sorted(vertical.items())])

t12, t13 = vertical[1] & vertical[2], vertical[1] & vertical[3]
show_table(["Tập", "Phép giao", "TID", "Support"], [
    ("{1,2}", "T(1) ∩ T(2)", sorted(t12), len(t12)),
    ("{1,3}", "T(1) ∩ T(3)", sorted(t13), len(t13)),
    ("{1,2,3}", "T(1,2) ∩ T(1,3)", sorted(t12 & t13), len(t12 & t13)),
])
sparse = (ROOT / "data/toy/sparse.txt").read_text(encoding="utf-8").splitlines()
singleton_counts = Counter(item for row in sparse for item in row.split())
print("Ở sparse.txt:", len(sparse), "giao dịch, count tối thiểu", ceil(Fraction("0.4") * len(sparse)),
      "và count singleton lớn nhất", max(singleton_counts.values()))

DFS xét các phần mở rộng theo ID tăng dần. Ở lớp tiền tố `(1)`, chỉ cần ghép `(1,2)` với sibling đứng sau là `(1,3)`. Tập `(1,2,3)` có support đúng bằng ngưỡng, nên vẫn được xuất. Mỗi itemset có một đường mở rộng tăng dần. Khi support của ứng viên dưới ngưỡng, mọi tập lớn hơn chứa nó cũng dưới ngưỡng. `sparse.txt` minh họa cắt ngay từ singleton.

## 2. Hai backend, một DFS

Basic giao hai danh sách TID tăng dần bằng hai con trỏ. Chi phí tối đa là $O(|T_X|+|T_Y|)$. BitVector dùng một bit mỗi TID, lấy AND từng từ máy và đếm số bit 1 (popcount). Chi phí giao và đếm là $O(\lceil n/64\rceil)$ từ máy trên máy đo. Mask rộng dù tập TID thưa, nên các danh sách ngắn có thể rẻ hơn. Cả hai tạo vùng nhớ mới cho tập con để lần ghép tiếp theo vẫn đọc đúng TID của cha.

Hai lệnh dưới gọi CLI thật. File xuất có dạng `1 2 #SUP: 3`, với ID theo thứ tự tăng dần và support nguyên.

In [ ]:
sys.path.insert(0, str(ROOT / "scripts"))
from benchlib import compare, read_patterns

for backend in ("list", "bitset"):
    run_julia("scripts/mine.jl", "--input", toy, "--output", OUTPUT / f"{backend}.txt",
              "--minsup", "40%", "--backend", backend)
basic = read_patterns(OUTPUT / "list.txt")
optimized = read_patterns(OUTPUT / "bitset.txt")
assert basic == optimized
show_table(["Itemset", "Support (giao dịch)"], basic.items())

## 3. Kiểm tính đúng

Chạy bộ test trong đề, gồm oracle vét cạn, các biên 63/64/65 và 127/128/129 giao dịch, ngưỡng đúng biên và các hàng trùng. Sau đó gọi Java SPMF trên năm toy. Bảng so **key itemset → support** với references đã kiểm bởi cả hai backend trong bộ Julia. Hai map có cùng số phần tử vẫn có thể sai một itemset hoặc sai support.

Input giữ ID dương, có thứ tự và phân tách bằng khoảng trắng. Output SPMF được chuẩn hóa thứ tự item để so theo key. Với benchmark và ứng dụng, tham số Java nằm bên trong khoảng cho cùng ngưỡng nguyên, tránh lỗi làm tròn ở biên phần trăm.

In [ ]:
run_julia("test/runtests.jl")

rows = []
for name in ("basic", "boundary", "dense", "duplicates", "sparse"):
    path = OUTPUT / f"toy-{name}-spmf.txt"
    run(JAVA, "-jar", ROOT / "tools/spmf.jar", "run", "Eclat", ROOT / f"data/toy/{name}.txt", path, "40%")
    counts = compare(read_patterns(ROOT / f"tests/reference/spmf/{name}.txt"), read_patterns(path))
    assert counts["identical"], (name, counts)
    rows.append((name, counts["reference_count"], counts["missing"], counts["extra"], counts["wrong_support"]))
assert read_patterns(OUTPUT / "toy-basic-spmf.txt") == basic
show_table(["Toy", "Số tập SPMF", "Thiếu", "Thừa", "Sai support"], rows)

## 4. Phép đo trên bốn bộ dữ liệu

Chess và Mushroom có giỏ dài, mật độ cao. Retail chứa nhiều mặt hàng nhưng mỗi giỏ trung bình chỉ khoảng 10 item. T10I4D100K có 100.000 giao dịch tổng hợp. Mật độ là số item trung bình mỗi giỏ chia tổng số item.

Mỗi bộ có sáu ngưỡng và năm lần đo. Retail còn được lấy tập con lồng nhau theo một hoán vị seed 42. Thí nghiệm độ rộng giữ nguyên số hàng và số item, dùng các tiền tố cùng hoán vị riêng cho từng hàng để so sánh ghép cặp.

- **End-to-end (giây)** đo mỗi lần trong tiến trình mới, gồm đọc/ghi, khởi động Julia hoặc JVM, JIT và khai thác. Đây là vùng so sánh cả ba backend.
- **Warm read + mine (ms)** đo riêng Julia sau warm-up, gồm đọc, dựng TID và khai thác. GC chủ động trước timer, ghi file và so map sau timer.
- **Peak RSS (MiB)** là đỉnh bộ nhớ cư trú toàn tiến trình. Byte cấp phát trong vùng Julia là chỉ số khác và có thể lớn hơn RSS.

Đồ thị dùng median và khoảng min–max năm lượt trên Mac M1, RAM 8 GiB. Ô dưới audit toàn bộ artifacts và chạy lại đối chiếu đủ map ở ngưỡng trung bình của bốn bộ. [Phương pháp](../docs/experiments.md) · [Nguồn FIMI](https://www.cs.rpi.edu/~zaki/Workshops/FIMI/data/)

In [ ]:
from IPython.display import Image, display
import json
from verify_delivery import check_report

run(sys.executable, "-B", "scripts/acquire_benchmarks.py")
run(sys.executable, "-B", "scripts/review_benchmarks.py", "--julia", JULIA, "--java", JAVA,
    "--output", OUTPUT / "benchmark-review")
report = check_report()
print("PASS PDF/source SHA-256:", report["pdf_sha256"])
evidence = json.loads((OUTPUT / "benchmark-review/review.json").read_text(encoding="utf-8"))
show_table(["Benchmark", "Min count", "Số tập SPMF", "Tỷ lệ khớp"],
           [(r["dataset"], r["min_count"], r["n_patterns"], f'{r["exact_match_rate"]:.0%}')
            for r in evidence["live_cases"]])
print("Bằng chứng:", OUTPUT / "benchmark-review/review.json")
stats = csv_rows(ROOT / "results/raw/dataset_statistics.csv")
show_table(["Tập", "Giao dịch", "Items", "Độ rộng TB", "Mật độ (%)"],
           [(r["dataset"], r["n_transactions"], r["n_items"], f'{float(r["average_width"]):.2f}',
             f'{100*float(r["density"]):.3f}') for r in stats])

plot_output = OUTPUT / "benchmark-figures"
run(sys.executable, "-B", "scripts/plot_results.py", "--output", plot_output)

### Runtime theo minsup và kích thước đầu ra

Xem panel trên của từng hình để so với SPMF cùng vùng đo. Ở Chess, giảm minsup làm số tập tăng rất mạnh, kéo chi phí giao và xuất kết quả lên. Mushroom cũng tăng ở vùng ngưỡng thấp. Retail thưa có ít tập ở ngưỡng cao, nhưng khi ngưỡng giảm thì output tăng rõ. Startup chiếm tỷ lệ đáng kể ở các trường hợp miner xử lý nhanh. Các đường BitVector ở scope warm cho thấy lợi ích giao bit rõ hơn trên Chess và T10I4D100K.

In [ ]:
for name in ("chess", "mushroom", "retail", "T10I4D100K"):
    display(Image(filename=str(plot_output / "plots" / f"runtime-{name}.png")))
display(Image(filename=str(plot_output / "plots/output-size.png")))

### RAM, scalability và độ rộng

Ở ngưỡng trung bình, Chess có tác vụ giao TID lớn nên dùng BitVector giúp cả warm median lẫn RSS. Retail có danh sách tương đối thưa và cả chi phí dựng mask, nên BitVector chậm hơn nhẹ ở phép đo warm. Với T10I4D100K, BitVector tăng tốc warm dù RSS nhỉnh hơn Basic.

Retail dùng các tập con 10/25/50/75/100% ở cùng ngưỡng tương đối 1%. Basic tăng đều hơn, BitVector chậm nhất ở 75% trong lượt đo này. Thay đổi số dòng cũng làm ngưỡng đếm và các tập đủ ngưỡng thay đổi. Đồ thị độ rộng tách số tập khỏi runtime. Width 25 và 30 có cùng số frequent itemsets trong phép thử, trong khi chi phí đọc và dựng TID vẫn có thể tăng.

In [ ]:
for name in ("memory", "scalability", "transaction-length"):
    display(Image(filename=str(plot_output / "plots" / f"{name}.png")))
speed = csv_rows(ROOT / "results/raw/optimization.csv")
show_table(["Tập", "Minsup", "Basic warm (ms)", "BitVector warm (ms)", "Basic/BitVector"],
           [(r["dataset"], r["minsup"], f'{float(r["basic_ms"]):.2f}', f'{float(r["bitvector_ms"]):.2f}',
             f'{float(r["speedup"]):.2f}×') for r in speed])

## Kết luận và cách đo lại

Hai biểu diễn trả cùng tập và support. Lợi ích tốc độ và RAM phụ thuộc vào độ dài TID list, ngưỡng và số tập đầu ra. Hai hướng thử tiếp là dùng list hoặc bitmap theo mật độ mỗi lớp tiền tố và gộp AND với popcount để giảm cấp phát. Số đo từ một máy, năm lượt và thứ tự cố định, nên cần đo lại nếu so sánh trên môi trường khác.

Các hình được vẽ vào thư mục của phiên chạy từ CSV đã kiểm hash. Để đo đầy đủ trên Linux/macOS, chạy `python3 scripts/benchmark_runner.py --output .tools/benchmark-new` sau khi lấy dữ liệu và JAR, rồi chạy `python3 scripts/plot_results.py --results .tools/benchmark-new`. [Lệnh đầy đủ](../README.md#benchmark)